### EOPF-SENTINEL-2 execution with Dask
This notebook demonstrates how to execute the EOPF-SENTINEL-2 application package using Calrissian on Kubernetes with task execution delegated to a Dask cluster.

Prepare a clean runtime environment by removing previous runs and results, then creating a fresh working directory.

In [ ]:
%%bash
rm -rf runs/
rm -rf /calrissian/results/*
mkdir runs/

Copy the CWL application package into the temporary runtime folder used for execution.

In [ ]:
%%bash
cp /workspace/dask-app-package/cwl-workflows/eopf-sentinel-2.cwl runs/

To allow Calrissian to locate the application package, copy it into the `/calrissian` mounted volume, which is shared with the Calrissian container.

In [ ]:
%%bash
cp runs/eopf-sentinel-2.cwl /calrissian/eopf-sentinel-2.cwl

Inspect the Kubernetes Job manifest that will be submitted to the cluster.

In [ ]:
%%bash
cat /workspace/dask-app-package/practice-lab/eopf-sentinel-2/k8s-job.yaml

Use `kubectl` to apply the Kubernetes Job definition and start the execution.

In [ ]:
%%bash
kubectl apply -f /workspace/dask-app-package/practice-lab/eopf-sentinel-2/k8s-job.yaml

Block execution until the Kubernetes Job completes or the timeout is reached.

In [ ]:
%%bash
kubectl wait --for=condition=complete --timeout=600s job/eopf-sentinel-2

List the generated output files and preview the resulting visualization.

In [ ]:
%%bash
RESULT_DIR=$(grep -oE '/calrissian/results/[a-zA-Z0-9]+' /calrissian/app.log | tail -n 1)

if [ -d "$RESULT_DIR" ]; then
  echo "Listing results in $RESULT_DIR"
  ls -lRah "$RESULT_DIR"/S2B_MSIL1C_20250113T103309_N0511_R108_T32TLQ_20250113T122458-processed
  echo
  echo '******************* catalog.json *******************'
  cat "$RESULT_DIR"/catalog.json
else
  echo "No results directory found"
fi
